# Who gains most from a coupon: an effect that changes with age

## At a glance

A retailer sent a discount coupon to some of its customers and wants to know whom to send
coupons to next time. The question is how the coupon's effect on the chance of a purchase
changes with the customer's age.

The estimated effect falls from about 24 percentage points at age 20 to about 3 points at 70. It
matches the simulation's true effect for young customers and runs high from 50 on: at 60 it
reads 7 points where the truth is 3. The decision it supports is to send coupons to customers
under about 58, 64% of them. The curve comes with no confidence interval, so held-out checks
stand in for one.

A coupon costs the profit of 0.08 purchases, so a customer's net gain is the rise in their
purchase chance minus 0.08. The data are simulated, so the page knows the true effect and can
score the rule: 0.0523 net purchases per customer, against 0.0546 for the best possible rule and
0.0293 for a coupon to everyone. With real data, estimate the rule's value instead, as the
section on deciding who gets a coupon shows.

## The question

The marketing team chose who got a coupon, and it sent more coupons to customers who spent more
last year, and slightly more to older customers. Those customers also buy more anyway. The question is, for customers of a given age,
how much a coupon raises the chance that they buy in the next month, compared with no coupon.
This is the conditional average treatment effect (CATE) given age: the average effect among
customers of that age.

A second, shorter analysis asks the same kind of question of a randomized drug trial: does the
drug's effect on the risk of an event change with how severe the patient's illness is?

## Setup

Every fit sets its seed, so a rerun prints the same numbers. Another version of `cleverly` can
print slightly different numbers.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

import cleverly

pd.set_option("display.width", 130)
print("cleverly", cleverly.__version__)

cleverly 0.2.0


## The data

The page simulates 2,000 customers, so it knows the true effect at every age. Real data never
show it.

In [2]:
def true_uplift(age):
    """The coupon's true effect on the chance of a purchase, by age."""
    return 0.25 / (1.0 + np.exp((age - 45.0) / 8.0))


def simulate_coupons(n, seed):
    rng = np.random.default_rng(seed)
    age = rng.uniform(18, 80, n)
    spend = rng.uniform(0, 1, n)  # last year's spend, scaled to 0-1
    visits = rng.poisson(3, n).clip(0, 10)  # store visits last month
    # The team sent more coupons to big spenders, and slightly more to older customers.
    chance = 1.0 / (1.0 + np.exp(-(-1.0 + 2.5 * spend + 0.02 * (age - 50))))
    coupon = rng.binomial(1, chance)
    buys = 0.15 + 0.4 * spend + 0.02 * visits + coupon * true_uplift(age)
    bought = rng.binomial(1, buys)
    return pd.DataFrame(
        {"age": age, "spend": spend, "visits": visits, "coupon": coupon, "bought": bought}
    )


# With your own data, load the data frame here, such as frame = pd.read_csv(...), and drop
# every line that uses true_uplift: real data have no known answer.
frame = simulate_coupons(n=2_000, seed=1)
print("rows and columns:", frame.shape)
print("column names:", list(frame.columns))
print(frame.head().round(3).to_string())

rows and columns: (2000, 5)
column names: ['age', 'spend', 'visits', 'coupon', 'bought']
      age  spend  visits  coupon  bought
0  49.733  0.284       5       0       1
1  76.929  0.010       4       0       0
2  26.938  0.110       4       0       1
3  76.816  0.131       2       0       0
4  37.334  0.432       2       1       1


Each row is one customer: age, last year's spend (scaled from 0 to 1), store visits last
month, whether they got a coupon, and whether they bought in the following month. The true
coupon effect falls smoothly with age, from about 24 percentage points for the youngest customers
to under 2 points after 65, and it does not depend on spend or visits.

## Why a simple comparison misleads

In [3]:
sent = frame["coupon"] == 1
print(f"share sent a coupon:          {sent.mean():.3f}")
print(
    f"mean spend, sent / not sent:  {frame.loc[sent, 'spend'].mean():.3f} / "
    f"{frame.loc[~sent, 'spend'].mean():.3f}"
)
print()
bands = pd.cut(frame["age"], [18, 35, 50, 65, 80], include_lowest=True)
naive = frame.groupby([bands, "coupon"], observed=True)["bought"].mean().unstack()
table = pd.DataFrame(
    {
        "simple difference": naive[1] - naive[0],
        "true effect": frame.groupby(bands, observed=True)["age"].apply(
            lambda ages: true_uplift(ages).mean()
        ),
    }
)
print(table.round(3).to_string())

share sent a coupon:          0.547
mean spend, sent / not sent:  0.579 / 0.390

                simple difference  true effect
age                                           
(17.999, 35.0]              0.328        0.224
(35.0, 50.0]                0.232        0.143
(50.0, 65.0]                0.126        0.047
(65.0, 80.0]                0.120        0.009


Customers who got a coupon spent more last year (0.579 against 0.390 on the scaled spend), and
big spenders buy more with or without a coupon. So within each age band, the simple difference
in purchase rates between customers with and without a coupon overstates the effect: 0.328
against a true 0.224 for the youngest band, and 0.120 against 0.009 for the oldest. Targeting
on these numbers would send coupons to older customers who gain almost nothing.

## State the question and what it needs

`CausalStudy` holds the data and the role of each column: the treatment (`coupon`), the outcome
(`bought`), and the columns to adjust for, which the models use to compare customers who are
alike (`age`, `spend`, `visits`). `Modifiers` names the column the effect may change with and the
range of ages the curve covers. `ConditionalContrast` is the question itself: the difference in
purchase chance with and without a coupon, as a function of age. `study.identify` returns the
conditions the answer rests on and the methods that can estimate it.

In [4]:
from cleverly import CausalStudy, ConditionalContrast, PointTreatment
from cleverly.bases import Modifiers

# For your own data, change the column names. `domains` gives the range the curve covers.
study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="bought", treatment="coupon", adjustment=("age", "spend", "visits")
    ),
)
by_age = Modifiers(["age"], domains={"age": (18, 80)})
effect = study.identify(ConditionalContrast(by_age))
# print(effect.summary(detail="full")) adds every condition in statistical notation and says
# why the other methods do not fit.
print(effect.summary())

                            Causal question (cate)
Question:      conditional average      Rows:                            2,000
               treatment effect given   Answered by:  adjusting for covariates
               age
Outcome:       bought (binary)
Treatment:     coupon (levels 0, 1)
Adjusted for:  age, spend, visits
Conditions the answer rests on
condition                checkable   what it says
------------------------------------------------------------------------------
well-defined treatment   no          Each unit's observed outcome is the
                                     outcome it would have had under the
                                     treatment it actually received. This
                                     needs the treatment to be well defined:
                                     every way of getting the same level has
                                     the same effect.
no interference          no          One unit's treatment does not change
         

The methods table names the one method that estimates a whole curve, `ep`. The answer rests on
these conditions:

- **No hidden confounding**: among customers of the same age, spend and visits, nothing else
  drove both the coupon and the purchase. The data cannot check it. The simulation makes it
  true; with your data it needs judgment about how the team chose.
- **Overlap**: every kind of customer had a real chance of getting a coupon and of not getting
  one. The data can show this but not prove it: the treatment model's predicted chances,
  printed after the fit, stay between 0.17 and 0.89.
- **Well-defined treatment** and **no interference**: a coupon means the same for everyone, and
  one customer's coupon does not change another's purchase. A coupon shared within a household
  would break the second. The data cannot check either.
- **A right treatment model or a right outcome model**: the method's correction, described in
  the next section, needs one of the two. The simulation assigns coupons through a logistic
  function of spend and age, so the logistic-regression treatment model here is right. With your
  data, use flexible learners for both.
- **A smooth curve**: the curve is built from a few smooth functions of age, so a sharp jump in
  the effect would be smoothed over.

## Estimate how the effect changes with age

The method is the efficient plug-in (EP) learner of van der Laan, Carone and Luedtke (2024). It fits an outcome model (who buys, given the coupon and the customer's columns) and a
treatment model (each customer's chance of a coupon), each row predicted by models trained
without it, like out-of-fold predictions in cross-validation. The outcome model alone gives an
effect for each customer, but if that model is wrong, so is the effect. The correction takes each
customer's prediction error (bought minus the outcome model's prediction), weights it by one over
the treatment model's chance of the option the customer got, and regresses the weighted errors on
a few smooth functions of age. Adding that fit to the outcome model's effect gives the curve.
When the treatment model is right, the weighted errors carry, on average, the part of the effect
the outcome model missed.

The smooth functions are cosine waves over the age range. The first is a single half-wave that
falls once from 18 to 80, and each further one adds a wiggle. Cross-validation chooses how many to
use.

In [5]:
# For your own data, swap in your own learners. Out-of-fold predictions keep a flexible
# learner honest, so a random forest or gradient boosting works here too.
curve = effect.estimate(
    "ep",
    outcome_learner=LogisticRegression(max_iter=1000),
    treatment_learner=LogisticRegression(max_iter=1000),
    random_state=0,
)
ages = np.array([20, 30, 40, 50, 60, 70, 80])
print(
    pd.DataFrame(
        {
            "age": ages,
            "estimated effect": curve.predict({"age": ages}),
            "true effect": true_uplift(ages),
        }
    )
    .round(3)
    .to_string(index=False)
)
print()
print("cosine waves the curve uses:", curve.sieve.chosen)
print("label:", curve.status)
chances = curve.diagnostics.support().propensity_quantiles["overall"]
print(f"predicted chance of a coupon: {chances[0.0]:.2f} to {chances[1.0]:.2f}")

 age  estimated effect  true effect
  20             0.239        0.239
  30             0.220        0.217
  40             0.177        0.163
  50             0.122        0.087
  60             0.069        0.033
  70             0.030        0.011
  80             0.016        0.003

cosine waves the curve uses: 1
label: experimental
predicted chance of a coupon: 0.17 to 0.89


The estimated curve matches the truth for young customers, 0.239 at age 20 where the truth is
0.239, and runs high from 50 on: 0.069 against 0.033 at 60, about double, and 0.030 against
0.011 at 70. Cross-validation chose one wave, the smoothest shape on offer. That half-wave passes
its midpoint at age 49, while the true effect falls fastest around 45, so the curve runs high in
the fifties and sixties.

The outcome model gives the coupon one coefficient, shared by every customer, so on its own it
misses how the effect changes with age. The treatment model is right here, and the correction
recovers most of the age pattern.

`cleverly` labels the fit experimental, because the method's theory does not cover a number of
waves chosen by cross-validation and no simulation study has yet checked the method. You can
report the curve if you say that it comes from a new method and has no interval. If you need
intervals, declare a simple shape for the effect, such as a line in age, and fit it with the
default method, as the [conditional effects guide](../user-guide/conditional-effects.md) shows.

## Check the curve on held-out rows

The data never show a customer's own effect, because each customer either got a coupon or did
not. The checks use a stand-in: the outcome model's predicted effect for that customer plus the
same weighted prediction error the correction uses. One customer's stand-in is very noisy, but
averaged over many customers it is right when either model is right. Each check splits the rows
into folds, refits the whole analysis without one fold, and compares the curve's predictions on
that fold with the stand-ins there.

In [6]:
calibration = curve.diagnostics.calibration()
print("folds:", calibration.n_folds)
print(f"calibration line: intercept {calibration.intercept:.3f}, slope {calibration.slope:.2f}")
bins = calibration.to_frame().rename(
    columns={"mean_prediction": "mean prediction", "pseudo_outcome_contrast": "mean stand-in"}
)
print(bins.round(3).to_string(index=False))
print()
risks = curve.diagnostics.risk_comparison().risks
print("out-of-fold error (lower is better):")
for name in ("curve", "dr_learner", "t_learner", "constant"):
    print(f"  {name:10s} {risks[name]:.4f}")

folds: 5
calibration line: intercept 0.028, slope 0.76
 mean prediction  mean stand-in  rows
           0.031          0.064   400
           0.065          0.097   400
           0.114          0.032   400
           0.183          0.191   400
           0.254          0.251   400



out-of-fold error (lower is better):
  curve      -0.0197
  dr_learner -0.0200
  t_learner  -0.0167
  constant   -0.0147


The calibration line regresses the stand-ins on the held-out predictions, and a perfect curve has
intercept 0 and slope 1. This line says the curve's differences between ages are somewhat too
large: where the curve predicts 0.03 the line gives about 0.05, and where it predicts 0.25 it
gives about 0.22. The five groups of 400 customers show how noisy the stand-ins are, since the
middle group reads 0.032 where the curve predicts 0.114. These are readings from one sample with
no interval, so read the curve's ranking of ages more firmly than its exact values.

The out-of-fold error scores each candidate against the stand-ins. It leaves out a term shared by
every candidate, so only differences mean anything and the numbers can be negative. The curve
(-0.0197) beats one effect for every age (`constant`, -0.0147) and the effect read straight off
the outcome model (`t_learner`, -0.0167). A flexible alternative that regresses the stand-ins on
age directly (`dr_learner`) scores about the same (-0.0200).

## Decide who gets a coupon

Suppose a purchase earns $50 of profit and a coupon costs $4, so a coupon costs the profit of
0.08 purchases. The rule sends a coupon to each customer whose estimated gain in purchase chance
is larger than that.

In [7]:
COST = 0.08
print(f"cost of one coupon, in purchases:  {COST:.2f}")
target = curve.predict({"age": frame["age"]}) > COST
print(f"share of customers targeted:      {target.mean():.3f}")
print(
    f"targeted ages:                    {frame.loc[target, 'age'].min():.0f} to "
    f"{frame.loc[target, 'age'].max():.0f}"
)
# The lines below score the rule with the true effect, which only simulated data have.
gain = true_uplift(frame["age"]) - COST
print(f"true net gain per customer, rule: {np.mean(gain * target):.4f}")
print(f"true net gain, best possible:     {np.mean(gain * (gain > 0)):.4f}")
print(f"true net gain, coupon for all:    {np.mean(gain):.4f}")

cost of one coupon, in purchases:  0.08
share of customers targeted:      0.643
targeted ages:                    18 to 58
true net gain per customer, rule: 0.0523
true net gain, best possible:     0.0546
true net gain, coupon for all:    0.0293


The rule earns almost all of the best possible net gain and much more than a coupon for everyone.
It stops too late: the true gain beats the cost only up to about age 51, and from 51 to 58 the
curve reads up to about twice the true effect, so those customers get coupons that lose a little
on average.

With real data, score the rule another way. Choose the cutoff age on one half
of the customers. On the other half, estimate the purchase rate under "a coupon if younger than the cutoff"
with a `Rule` and `RegimeContrast` (see
[known regimes](../user-guide/estimands.md#known-regimes)), which gives an interval, and subtract
the cost of the coupons the rule sends. A randomized holdout checks it best: coupons by the rule
for a random share of customers, current practice for another.

## A trial: does the drug help sicker patients more?

This example shows two things the coupon data could not: an effect measured as a ratio, and a
treatment model known to be right because the drug was assigned at random. In a simulated trial
of 2,000 patients, the question is how the risk ratio (the risk of an event with the drug divided
by the risk without it) changes with a severity score from 0 to 1.

`Cosine(["severity"], 6)` offers up to six cosine waves in severity, and `Sieve(..., size=1)`
fixes the curve to the first of them before the fit instead of letting cross-validation choose,
as a trial's analysis plan would fix it in advance. Adjusting for severity and age is not needed
against confounding in a trial, but it can make the estimate more precise. On this scale
`predict` returns the log of the risk ratio, so the code applies `np.exp`.

In [8]:
from cleverly.bases import Cosine, Sieve


def simulate_trial(n, seed):
    rng = np.random.default_rng(seed)
    severity = rng.uniform(0, 1, n)  # a baseline severity score, 0-1
    age = rng.uniform(40, 80, n)
    drug = rng.binomial(1, 0.5, n)  # assigned at random
    risk = 0.20 + 0.40 * severity + 0.002 * (age - 60)
    event = rng.binomial(1, risk * np.exp(drug * (-0.8 * severity)))
    return pd.DataFrame({"severity": severity, "age": age, "drug": drug, "event": event})


trial = simulate_trial(n=2_000, seed=1)
trial_effect = CausalStudy(
    trial,
    design=PointTreatment(outcome="event", treatment="drug", adjustment=("severity", "age")),
).identify(
    ConditionalContrast(Modifiers(["severity"], domains={"severity": (0, 1)}), scale="log_ratio")
)
ratio_curve = trial_effect.estimate(
    "ep",
    outcome_learner=LogisticRegression(max_iter=1000),
    treatment_learner=LogisticRegression(max_iter=1000),
    sieve=Sieve(Cosine(["severity"], 6), size=1),
    random_state=0,
)
levels = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
print(
    pd.DataFrame(
        {
            "severity": levels,
            "estimated risk ratio": np.exp(ratio_curve.predict({"severity": levels})),
            "true risk ratio": np.exp(-0.8 * levels),
        }
    )
    .round(2)
    .to_string(index=False)
)
trial_calibration = ratio_curve.diagnostics.calibration()
print()
print(f"calibration slope: {trial_calibration.slope:.2f}")

 severity  estimated risk ratio  true risk ratio
     0.00                  1.03             1.00
     0.25                  0.92             0.82
     0.50                  0.70             0.67
     0.75                  0.54             0.55
     1.00                  0.48             0.45



calibration slope: 0.89


The estimated risk ratio falls from 1.03 for the least severe patients, where the drug does
nothing, to 0.48 for the most severe, against the true 0.45. Its largest miss is at severity 0.25
(0.92 against 0.82). The calibration slope of 0.89 is near 1.

## What we can and can't conclude

For the marketing team: the coupon's effect falls with age. Coupons for customers under about
58 come close to the best possible rule in this simulation and earn nearly twice the gain of a
coupon for everyone, though the rule reaches a few years too old. That rests on the conditions
listed under "State the question and what it needs", above all that age, spend and visits account
for how the team chose who got a coupon. The held-out checks show the curve predicts better than
one effect for all ages and better than the outcome model alone; they give no range for its
values.

The page does not show that the estimated curve is the true curve for your data, that a rule
learned on one sample of customers earns the same on the next, or anything about how the method
performs in general.

## Go deeper

### What has been checked so far

The package's tests check the correction exactly on small made-up populations whose answer is
known, including when the outcome model is wrong. Three simulation studies of the method are
planned and have not run; the
[conditional effects reference](../technical-reference/conditional-effects.md) states what they
will measure and what the theory claims.

### Does the curve depend on how the rows were split?

The out-of-fold fits split the rows at random. `stability()` refits the curve under five
different splits of the same rows and compares the spread of those refits with the spread of the curve
across ages.

In [9]:
stability = curve.diagnostics.stability()
print(f"spread across 5 refits, relative to the curve's own spread: {stability.relative_sd:.3f}")
print("warning:", stability.warning)

spread across 5 refits, relative to the curve's own spread: 0.208


The refits move by about a fifth (0.208) of the curve's own spread, well below the level at which
`cleverly` warns (a half). This measures sensitivity to the split on the same data, not sampling
noise. A value above a half would mean the curve depends on how the rows happen to be split, and
a simpler shape or more data would be the next step. A low value, as here, rules that out but does
not show that the curve is well determined.

## Where to go next

The [conditional effects guide](../user-guide/conditional-effects.md) covers the simple shapes
with intervals. The [one treatment, one outcome](point-treatment-tmle.ipynb)
example explains the outcome and treatment models step by step, and
[who should get the offer](treatment-rules.ipynb) learns a treatment rule directly. The
[examples index](index.md) lists every example.